<!-- nav -->
[← 9 · Constraints as data](09_Constraints_As_Data.ipynb) · [Home](../../README.md) · [11 · Constraints over collections →](11_Constraints_Over_Collections.ipynb)

# 10 · Answers you can trust

**The problem.** Loggers drop out: a battery dies, a radio loses signal, and a shipment arrives with no temperature on
file. What should "is it in range?" answer then? A Python function raises `TypeError` comparing `None` with `2.0`. A
JavaScript one compares `undefined` and says `false`: the shipment is quietly rejected, or, with the test written the
other way round, quietly accepted. Neither is an answer you can act on.

An expression answers **unknown** when the data doesn't decide, and says so. This case study shows how unknowns
combine, how you decide what they mean, and why `1` and `1.0` aren't equal.

In [1]:
from mbse.Expressions.Patterns.Expressions import Simplifiers, Expressions as E
from toolkit import cold_chain, show

store = cold_chain()
Expression = store.Expressions.Framework.Expression
with store.transaction():
    s1 = store.Shipment(lambda s: s.id("S1").celsius(5.5).hours(30))
    s2 = store.Shipment(lambda s: s.id("S2").celsius(9.1).hours(80))
    s3 = store.Shipment(lambda s: s.id("S3").hours(20))  # its logger never reported
shipments = [s1, s2, s3]
shipment = E.variable("shipment")
celsius = shipment.celsius
between = E.literal(2.0).le(celsius).and_(celsius.le(8.0))  # a writer, which other writers build on
with store.transaction():
    in_range = Expression(lambda e: e.root(between))


def table(constraint):
    return {s.id.get(): Simplifiers.OfAny(constraint, {"shipment": s}).evaluate() for s in shipments}

<a id="T10.1"></a>
## 10.1 · Absent is unknown

In mbse-schemas, a property is either present or absent; there's no `None` value to compare. Reading an absent
property gives **unknown**, which Python writes `None`, and a comparison with unknown is unknown:

In [2]:
table(in_range)

{'S1': True, 'S2': False, 'S3': None}

S3's answer isn't false: it's that nobody knows. `has` asks whether a property is present, which always has an
answer:

In [3]:
with store.transaction():
    reported = Expression(lambda e: e.root(shipment.has("celsius")))
print(show(reported))
table(reported)

has(shipment, 'celsius')


{'S1': True, 'S2': True, 'S3': False}

<a id="T10.2"></a>
## 10.2 · How unknowns combine

`and`, `or`, `not` and `implies` follow **Kleene's three-valued logic**: an unknown operand doesn't matter when the
other one decides. `False and unknown` is false, whatever the unknown turns out to be; `True or unknown` is true.

In [4]:
with store.transaction():
    late_or_out = Expression(lambda e: e.root(shipment.hours.gt(72).or_(between.not_())))
    ok_or_silent = Expression(lambda e: e.root(shipment.has("celsius").not_().or_(between)))
for constraint in (late_or_out, ok_or_silent):
    print(show(constraint))
    print(table(constraint))

or(gt(get(shipment, 'hours'), 72), not(and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0))))
{'S1': False, 'S2': True, 'S3': None}
or(not(has(shipment, 'celsius')), and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0)))
{'S1': True, 'S2': False, 'S3': True}


For S3, the first constraint is unknown: it was on time, so everything rests on a temperature nobody has. The second
decides S3 at its first operand, and, as in Python, doesn't evaluate the second.

`implies` reads "if ... then": a constraint about only some shipments holds trivially for the others:

In [5]:
with store.transaction():
    long_trips_cold = Expression(lambda e: e.root(shipment.hours.gt(72).implies(shipment.celsius.le(8.0))))
print(show(long_trips_cold))
table(long_trips_cold)

implies(gt(get(shipment, 'hours'), 72), le(get(shipment, 'celsius'), 8.0))


{'S1': True, 'S2': False, 'S3': True}

<a id="T10.3"></a>
## 10.3 · You decide what unknown means

An evaluator never guesses. The program asking decides, once, at the point where it acts. A dashboard that flags
shipments to inspect flags what isn't known to be fine; an audit that certifies shipments certifies only what is known
to be fine:

In [6]:
inspect = [s.id.get() for s in shipments if Simplifiers.OfAny(in_range, {"shipment": s}).evaluate() is not True]
certify = [s.id.get() for s in shipments if Simplifiers.OfAny(in_range, {"shipment": s}).evaluate() is True]
inspect, certify

(['S2', 'S3'], ['S1'])

Writing `if Evaluators.OfAny(...)` would treat unknown as false, which is the silent mistake this case study
started with. Compare with `is True` or `is False`.

<a id="T10.4"></a>
## 10.4 · No coercion

An `int` and a `float` are different kinds of value, as they are in mbse-schemas, of different domains, as
[tutorial 2](02_One_Value_A_Domain_In_Each_Language.ipynb) shows: `1 == 1.0` isn't true, it's unknown, because the two aren't comparable. That catches a classic slip. `celsius` is a float, so a constraint that
writes the limit as an int answers unknown for every shipment, rather than an answer that depends on the language:

In [7]:
with store.transaction():
    sloppy = Expression(lambda e: e.root(E.literal(2).le(shipment.celsius).and_(shipment.celsius.le(8))))
table(sloppy)

{'S1': None, 'S2': None, 'S3': None}

Python would have compared `2 <= 5.5` happily, and TypeScript compares an integer `2n` with `5.5` too, but by its own
rules. A constraint meant to give the same answer everywhere can't depend on either. Arithmetic is stricter still:
adding an int and a float is an error, raised when the constraint is evaluated:

In [8]:
with store.transaction():
    mixed = Expression(lambda e: e.root(shipment.hours.add(0.5).gt(72.0)))
try:
    Simplifiers.OfAny(mixed, {"shipment": s1}).evaluate()
except TypeError as error:
    print(error)

add expects numbers of one domain, got int and float


<a id="T10.5"></a>
## 10.5 · What raises

Unknown is for data that doesn't decide. A constraint that can't be evaluated at all raises: a variable nothing binds,
an operation Patterns doesn't define, an operand of the wrong kind. Those are mistakes in the constraint, not in the
data, and `validate` (case study 9) reports the first two before the constraint is evaluated.

In [9]:
with store.transaction():
    divided = Expression(lambda e: e.root(E.operation("div", 1, 2)))
    numbered = Expression(lambda e: e.root(shipment.id.add(1)))
for expression, scope in [(in_range, {}), (divided, {}), (numbered, {"shipment": s1})]:
    try:
        Simplifiers.OfAny(expression, scope).evaluate()
    except Exception as error:
        print(type(error).__name__, error)

KeyError "variable 'shipment' is not bound"
NotImplementedError 'div' is not an operation Patterns defines
TypeError add expects numbers of one domain, got str and int


## Why it works this way

- **Unknown is not false.** Treating it as false fails good records silently, and treating it as true passes bad ones,
  so the program asking decides what it means
  ([PRI-007: three-valued checking on request](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-007)).
- **Kleene, because it's the logic of not knowing yet.** `False and unknown` is false because no value of the unknown
  could make it true: more data can't overturn an answer an expression gives.
- **No coercion, because languages disagree.** Python, TypeScript, MATLAB, Excel and C each convert numbers by their own rules,
  and a constraint must mean the same everywhere.

How expressions evaluate, in full:
[CAP-106: expressions evaluated three-valued](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-106.md#evaluation-in-mbse-expressions).

<!-- nav -->
[← 9 · Constraints as data](09_Constraints_As_Data.ipynb) · [Home](../../README.md) · [11 · Constraints over collections →](11_Constraints_Over_Collections.ipynb)